# 📉 A empresa continua saudável. Então por que o lucro está desaparecendo?
### A matemática que a maioria dos empresários não vê vindo

**A tese deste notebook:** muitos empresários tratam dívida como algo neutro ou até vantajoso, porque "o juro é dedutível do Imposto de Renda". Essa frase é *verdadeira*, mas incompleta — e a parte que falta é exatamente a que quebra empresas.

**A Selic não precisa destruir a operação de uma empresa para destruir seu lucro. Basta que o custo da dívida consuma o resultado operacional.**

Vamos simular uma empresa fictícia, saudável na operação (a mesma empresa, com o mesmo EBIT, o tempo todo), e variar apenas a Selic. O objetivo é mostrar, com números e gráficos, como a despesa financeira de uma dívida atrelada a CDI cresce de forma linear com a Selic — e como essa despesa, ao ser subtraída do EBIT para chegar ao lucro antes do IR, faz o **lucro líquido** desabar de forma muito mais brutal do que a própria alta de juros. A Selic não altera o EBIT em nenhum momento — ela consome o que vem depois dele.

> ⚠️ **Aviso:** este é um exercício educacional com uma empresa fictícia e premissas simplificadas — o custo da dívida, a alíquota efetiva de IR/CSLL e o aproveitamento do tax shield estão declarados explicitamente na seção 3. Não é recomendação de investimento nem substitui uma análise financeira real (que também consideraria capital de giro, outras receitas/despesas, covenants, hedge, etc.).

## 0. Configuração do ambiente

In [ ]:
import datetime
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import requests

try:
    import plotly.express as px
except ImportError:
    !pip install plotly -q
    import plotly.express as px

pd.options.display.float_format = lambda x: f"{x:,.2f}"
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

print("Ambiente pronto.")

## 1. Dados reais: série histórica da Selic (Banco Central do Brasil)

Usamos a API pública de **Dados Abertos do Banco Central** (SGS — Sistema Gerenciador de Séries Temporais), série **432 — Meta Selic definida pelo Copom (% a.a.)**.

Documentação da série: https://dadosabertos.bcb.gov.br/dataset/432-taxa-de-juros---selic

⚠️ **Importante:** desde 26/03/2025 o BCB **limita cada consulta a no máximo 10 anos** de intervalo — pedir um período maior retorna erro (na prática, um `406 Not Acceptable`). Por isso a função abaixo quebra o período pedido em janelas de até 10 anos, consulta cada uma separadamente e depois concatena tudo. Se mesmo assim a API estiver fora do ar, cai para uma pequena amostra offline, só para o notebook não travar.

In [ ]:
def _gerar_janelas(data_inicio, data_fim, anos_max=5):
    """Quebra o intervalo em janelas de no máximo `anos_max` anos (limite do BCB desde mar/2025).
    Usamos janelas de 5 anos (menores que o limite de 10) para reduzir o tamanho de cada
    resposta e evitar timeout, já que a série retorna um valor por dia."""
    janelas = []
    ini = data_inicio
    while ini < data_fim:
        fim_janela = min(ini + datetime.timedelta(days=365 * anos_max), data_fim)
        janelas.append((ini, fim_janela))
        ini = fim_janela + datetime.timedelta(days=1)
    return janelas


def buscar_selic_bcb(data_inicio="01/01/2003", data_fim=None, timeout=60, tentativas=3):
    """Busca a série histórica da Meta Selic (série 432) na API do BCB (SGS),
    respeitando o limite de 10 anos por consulta e com novas tentativas em caso de timeout."""
    data_fim_dt = (datetime.date.today() if data_fim is None
                   else datetime.datetime.strptime(data_fim, "%d/%m/%Y").date())
    data_inicio_dt = datetime.datetime.strptime(data_inicio, "%d/%m/%Y").date()

    partes = []
    for ini, fim in _gerar_janelas(data_inicio_dt, data_fim_dt, anos_max=5):
        url = (
            "https://api.bcb.gov.br/dados/serie/bcdata.sgs.432/dados"
            f"?formato=json&dataInicial={ini.strftime('%d/%m/%Y')}&dataFinal={fim.strftime('%d/%m/%Y')}"
        )
        for tentativa in range(1, tentativas + 1):
            try:
                resp = requests.get(url, timeout=timeout)
                resp.raise_for_status()
                partes.append(pd.DataFrame(resp.json()))
                break
            except Exception as e:
                if tentativa == tentativas:
                    print(f"⚠️ Falha na janela {ini:%d/%m/%Y}–{fim:%d/%m/%Y} "
                          f"após {tentativas} tentativas: {e}")
                else:
                    time.sleep(2 * tentativa)

    if not partes:
        print("⚠️ Não foi possível acessar a API do BCB em nenhuma janela.")
        print("   Usando uma amostra offline apenas para não travar o notebook.")
        amostra = {
            "data": ["2015-01-01", "2016-07-01", "2018-01-01", "2020-08-01",
                     "2021-12-01", "2022-08-01", "2023-08-01", "2024-12-01", "2025-12-01"],
            "selic_meta_%": [11.75, 14.25, 6.75, 2.00, 9.25, 13.75, 13.25, 12.25, 15.00],
        }
        df = pd.DataFrame(amostra)
        df["data"] = pd.to_datetime(df["data"])
        return df

    df = pd.concat(partes, ignore_index=True)
    df["data"] = pd.to_datetime(df["data"], format="%d/%m/%Y")
    df["valor"] = df["valor"].astype(float)
    df = (df.rename(columns={"valor": "selic_meta_%"})
            .drop_duplicates("data")
            .sort_values("data")
            .reset_index(drop=True))
    print(f"✅ {len(df)} pontos carregados da API do Banco Central (SGS, série 432), "
          f"de {df['data'].min():%d/%m/%Y} a {df['data'].max():%d/%m/%Y}.")
    return df

df_selic_hist = buscar_selic_bcb()
df_selic_hist.tail(10)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(df_selic_hist["data"], df_selic_hist["selic_meta_%"], color="#0b5394", linewidth=1.8)
ax.fill_between(df_selic_hist["data"], df_selic_hist["selic_meta_%"], color="#0b5394", alpha=0.08)
ax.set_title("Meta Selic definida pelo Copom — série histórica (BCB/SGS 432)", fontsize=13, weight="bold")
ax.set_ylabel("Selic (% a.a.)")
ax.set_xlabel("")
plt.tight_layout()
plt.show()

## 2. A "ilusão" do juro dedutível

O argumento mais comum a favor de se endividar é: *"a despesa financeira reduz a base de cálculo do IR/CSLL, então uma parte do juro 'volta' em imposto que eu deixo de pagar"*. Isso é matematicamente verdade — **enquanto a empresa ainda tem lucro tributável**.

O problema é o que acontece quando a despesa financeira cresce o suficiente para **consumir todo o lucro operacional (EBIT)**:

- Enquanto há lucro antes do IR, cada real de despesa financeira adicional custa, líquido de imposto, `(1 - alíquota)` reais de lucro — no nosso exemplo, ~66 centavos por real de juro.
- No momento em que a despesa financeira ultrapassa o EBIT, a empresa passa a ter **prejuízo**. Não há mais imposto a economizar (não existe alíquota negativa) — cada real adicional de juro agora custa o real **inteiro**, não mais 66 centavos.
- E o mais importante: a dívida continua vencendo, com juros e principal, **independentemente de a empresa ter lucro ou prejuízo**. O escudo fiscal nunca foi de graça — ele só existia enquanto havia lucro para escudar.

> 📎 **Premissa assumida:** para os 66 centavos acima, consideramos alíquota efetiva de IR/CSLL de **34%** e **aproveitamento integral e imediato** do benefício fiscal, sempre que há lucro tributável no período. Empresas em lucro presumido, com prejuízo fiscal acumulado, incentivos setoriais ou outro regime tributário podem ter um aproveitamento diferente — o modelo é didático, não universal.

É esse ponto de virada que a simulação abaixo torna visível.

## 3. A empresa fictícia e suas premissas

Criamos uma empresa fictícia, operacionalmente estável (o EBIT não muda em nenhum cenário), para isolar exatamente o efeito da Selic sobre o resultado financeiro.

**Premissas explícitas** (todas ajustáveis nos parâmetros abaixo — mude e rode de novo com os números da sua empresa):

- A despesa financeira **não** é `Dívida × Selic` pura. É `Dívida × (CDI + spread bancário)`, com **CDI ≈ Selic − 0,10 p.p.** (aproximação usual de mercado) e **spread bancário de 2 p.p.**
- Na prática, isso equivale a um **custo efetivo de dívida ≈ Selic + 1,9 p.p.** — o spread de 1,9 p.p. é uma escolha de modelagem (uma dívida CDI+ real pode ter spread maior, menor, ou a empresa pode ter parte da dívida prefixada), não uma regra universal.
- Assumimos alíquota efetiva combinada de IR/CSLL de **34%**, com aproveitamento integral do benefício fiscal enquanto houver lucro tributável (ver premissa detalhada na seção 2).

In [ ]:
# ----- Parâmetros da empresa fictícia -----
RECEITA_LIQUIDA   = 100_000_000   # R$ 100 milhões / ano
MARGEM_EBIT       = 0.20          # EBIT = 20% da receita (estável, não depende da Selic)
EBIT              = RECEITA_LIQUIDA * MARGEM_EBIT

DIVIDA_TOTAL      = 80_000_000    # R$ 80 milhões, 100% atrelada a CDI
SPREAD_BANCARIO   = 0.02          # +2 p.p. de spread sobre o CDI
DESCONTO_CDI_SELIC = 0.001        # CDI ≈ Selic - 0,10 p.p. (aproximação usual no mercado)
ALIQUOTA_IR_CSLL  = 0.34          # IRPJ + CSLL combinados (regime de lucro real)

# spread líquido efetivo sobre a própria Selic: (Selic - desconto_cdi) + spread = Selic + (spread - desconto_cdi)
SPREAD_LIQUIDO_SOBRE_SELIC = SPREAD_BANCARIO - DESCONTO_CDI_SELIC

print(f"Receita líquida anual: R$ {RECEITA_LIQUIDA:,.0f}")
print(f"EBIT (lucro operacional): R$ {EBIT:,.0f}  ({MARGEM_EBIT:.0%} da receita)")
print(f"Dívida total (100% CDI+): R$ {DIVIDA_TOTAL:,.0f}")
print(f"Alavancagem (Dívida/Receita): {DIVIDA_TOTAL/RECEITA_LIQUIDA:.0%}")
print(f"👉 Custo efetivo da dívida ≈ Selic + {SPREAD_LIQUIDO_SOBRE_SELIC:.1%} "
      f"(CDI = Selic − {DESCONTO_CDI_SELIC:.1%}, spread bancário = {SPREAD_BANCARIO:.1%})")
print(f"   Ou seja: Despesa Financeira ≈ R$ {DIVIDA_TOTAL:,.0f} × (Selic + {SPREAD_LIQUIDO_SOBRE_SELIC:.1%}), "
      f"NÃO R$ {DIVIDA_TOTAL:,.0f} × Selic puro.")

In [ ]:
def calcular_dre(selic,
                  divida=DIVIDA_TOTAL,
                  ebit=EBIT,
                  spread=SPREAD_BANCARIO,
                  desconto_cdi=DESCONTO_CDI_SELIC,
                  aliquota_ir=ALIQUOTA_IR_CSLL):
    """
    DRE simplificado de uma empresa cuja dívida é 100% atrelada a CDI + spread.
    Fórmula-chave: despesa financeira = Dívida x (CDI + spread) = Dívida x (Selic - desconto + spread)
                 = Dívida x (Selic + spread_líquido), com spread_líquido = spread - desconto_cdi (≈ 1,9 p.p. nos parâmetros padrão)
    Note que a Selic entra apenas no cálculo da despesa financeira — o EBIT nunca é alterado por ela.
    """
    cdi = selic - desconto_cdi
    custo_da_divida = cdi + spread
    despesa_financeira = divida * custo_da_divida

    lair = ebit - despesa_financeira  # Lucro Antes do IR

    if lair > 0:
        ir_csll = lair * aliquota_ir
        lucro_liquido = lair - ir_csll
    else:
        ir_csll = 0.0          # sem lucro tributável, sem escudo fiscal adicional
        lucro_liquido = lair   # prejuízo líquido = prejuízo antes do IR

    return {
        "Selic (% a.a.)": round(selic * 100, 2),
        "EBIT (R$)": ebit,
        "Despesa Financeira (R$)": despesa_financeira,
        "Lucro Antes do IR (R$)": lair,
        "IR/CSLL (R$)": ir_csll,
        "Lucro Líquido (R$)": lucro_liquido,
        "Margem Líquida (%)": round((lucro_liquido / RECEITA_LIQUIDA) * 100, 2),
    }

# teste rápido
calcular_dre(0.15)

## 4. Tabela comparativa: 3 cenários de Selic

Rodamos o mesmo DRE para **Selic a 10%, 15% e 20%** — a única variável que muda entre os cenários é a Selic.

In [ ]:
cenarios = [0.10, 0.15, 0.20]
df_cenarios = pd.DataFrame([calcular_dre(s) for s in cenarios])
df_cenarios.index = [f"Selic {s:.0%}" for s in cenarios]

colunas_rs = ["EBIT (R$)", "Despesa Financeira (R$)", "Lucro Antes do IR (R$)",
              "IR/CSLL (R$)", "Lucro Líquido (R$)"]

(df_cenarios.style
    .format({c: "R$ {:,.0f}".format for c in colunas_rs})
    .format({"Selic (% a.a.)": "{:.1f}%", "Margem Líquida (%)": "{:.2f}%"})
    .background_gradient(subset=["Lucro Líquido (R$)"], cmap="RdYlGn")
    .set_caption("DRE simulado — mesma empresa, mesmo EBIT, apenas a Selic muda"))

In [ ]:
queda_10_para_20 = df_cenarios.loc["Selic 10%", "Lucro Líquido (R$)"] - df_cenarios.loc["Selic 20%", "Lucro Líquido (R$)"]
queda_pct = queda_10_para_20 / df_cenarios.loc["Selic 10%", "Lucro Líquido (R$)"]

print("📌 Leitura do resultado:")
print(f"- Despesa financeira quase dobra entre Selic 10% e 20% "
      f"(de R$ {df_cenarios.loc['Selic 10%','Despesa Financeira (R$)']:,.0f} "
      f"para R$ {df_cenarios.loc['Selic 20%','Despesa Financeira (R$)']:,.0f}).")
print(f"- O EBIT (lucro operacional) NÃO mudou em nenhum cenário: "
      f"R$ {EBIT:,.0f} nos três casos.")
print(f"- Mesmo assim, o lucro líquido caiu R$ {queda_10_para_20:,.0f} "
      f"({queda_pct:.0%}) só por causa da Selic subir de 10% para 20%.")
print("- A empresa continua vendendo, produzindo e operando exatamente igual.")
print("  O que quebrou o resultado foi 100% financeiro, não operacional.")

## 5. Simulação contínua: da Selic a 5% até 25%

Agora vamos além dos 3 cenários e simulamos uma faixa contínua de Selic, para ver a curva completa — e o ponto exato em que a despesa financeira **desta empresa** (R$ 80 milhões de dívida, custo ≈ Selic + 1,9 p.p.) passa a consumir 100% do seu EBIT. Chamamos esse ponto de **"ponto de cobertura do custo da dívida sobre o EBIT"**.

⚠️ Esse ponto é específico **desta combinação de alavancagem e spread** — não é um limite geral de Selic válido para qualquer empresa endividada. Uma empresa com menos dívida, ou com spread menor, teria esse ponto num patamar de Selic mais alto; uma mais alavancada, num patamar mais baixo.

In [ ]:
selic_range = np.round(np.arange(0.05, 0.2501, 0.0025), 4)
df_simulacao = pd.DataFrame([calcular_dre(s) for s in selic_range])

# Selic em que a despesa financeira DESTA empresa consome 100% do seu EBIT:
# EBIT = Dívida x (Selic - desconto_cdi + spread)  =>  Selic = EBIT/Dívida - spread + desconto_cdi
selic_cobertura_ebit = EBIT / DIVIDA_TOTAL - SPREAD_BANCARIO + DESCONTO_CDI_SELIC

print(f"👉 Com Selic acima de {selic_cobertura_ebit:.1%}, a despesa financeira desta empresa "
      f"(dívida de R$ {DIVIDA_TOTAL/1e6:.0f} mi a custo ≈ Selic + {SPREAD_LIQUIDO_SOBRE_SELIC:.1%}) "
      f"já consome 100% do seu EBIT — mesmo que a operação continue saudável.")
print("   Esse patamar muda se você alterar a dívida, o spread ou o EBIT nos parâmetros da seção 3.")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

ax.plot(df_simulacao["Selic (% a.a.)"], df_simulacao["Lucro Líquido (R$)"] / 1e6,
        color="#1f6f43", linewidth=2.5, label="Lucro Líquido")
ax.axhline(0, color="black", linewidth=1)
ax.axvline(selic_cobertura_ebit * 100, color="crimson", linestyle="--", linewidth=1.5,
           label=f"Cobertura do custo da dívida sobre o EBIT ({selic_cobertura_ebit:.1%})")

ax.fill_between(df_simulacao["Selic (% a.a.)"], df_simulacao["Lucro Líquido (R$)"] / 1e6, 0,
                where=(df_simulacao["Lucro Líquido (R$)"] >= 0), color="#1f6f43", alpha=0.12)
ax.fill_between(df_simulacao["Selic (% a.a.)"], df_simulacao["Lucro Líquido (R$)"] / 1e6, 0,
                where=(df_simulacao["Lucro Líquido (R$)"] < 0), color="crimson", alpha=0.12)

for s in [10, 15, 20]:
    r = calcular_dre(s / 100)
    ax.scatter([s], [r["Lucro Líquido (R$)"] / 1e6], color="darkorange", zorder=5, s=70,
               edgecolor="black", linewidth=0.8)
    ax.annotate(f"Selic {s}%\nR$ {r['Lucro Líquido (R$)']/1e6:.1f}mi",
                (s, r["Lucro Líquido (R$)"] / 1e6), textcoords="offset points",
                xytext=(0, 12), ha="center", fontsize=9)

ax.set_xlabel("Selic (% a.a.)")
ax.set_ylabel("Lucro Líquido (R$ milhões)")
ax.set_title("Resultado líquido despenca conforme a Selic sobe\n(EBIT constante em todos os cenários)",
             fontsize=13, weight="bold")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

## 6. Quanto lucro desaparece a cada 1 ponto percentual de alta

Dentro deste modelo, a perda de lucro líquido por ponto percentual de Selic é **constante dentro de cada regime tributário** — e existem exatamente dois regimes, não uma curva que acelera continuamente:

- **Com lucro tributável:** cada +1 p.p. de Selic reduz o lucro líquido em `Dívida × 1 p.p. × (1 − alíquota)` — o escudo fiscal amortece parte do golpe.
- **Já em prejuízo:** o escudo fiscal desaparece (não existe alíquota negativa) e cada +1 p.p. reduz o lucro líquido no valor **cheio**: `Dívida × 1 p.p.`

Ou seja: a Selic não faz o estrago acelerar aos poucos — ela faz o estrago **saltar de patamar**, de uma vez, exatamente no momento em que o lucro tributável acaba. É uma quebra de regime, não uma curva.

In [ ]:
selic_pp = np.arange(5, 26, 1) / 100
df_pp = pd.DataFrame([calcular_dre(s) for s in selic_pp])
df_pp["Perda de Lucro Líquido vs. ponto anterior (R$)"] = -df_pp["Lucro Líquido (R$)"].diff()

cores = ["#1f6f43" if s <= selic_cobertura_ebit * 100 else "crimson" for s in df_pp["Selic (% a.a.)"]]

fig, ax = plt.subplots(figsize=(11, 5.5))
ax.bar(df_pp["Selic (% a.a.)"][1:], df_pp["Perda de Lucro Líquido vs. ponto anterior (R$)"][1:] / 1e3,
       color=cores[1:], width=0.8)
ax.axvline(selic_cobertura_ebit * 100, color="black", linestyle=":", linewidth=1.2)
ax.set_xlabel("Selic (% a.a.)")
ax.set_ylabel("Lucro Líquido perdido no ponto (R$ mil)")
ax.set_title("Perda de lucro líquido a cada +1 p.p. de Selic — dois patamares constantes\n"
             "(verde: com escudo fiscal · vermelho: sem escudo fiscal, empresa já no prejuízo)",
             fontsize=12, weight="bold")
plt.tight_layout()
plt.show()

# Valores teóricos exatos de cada regime — são constantes por definição do modelo
# (o único ponto que foge do padrão no gráfico é a barra que atravessa a própria transição,
#  onde a perda é uma mistura dos dois regimes)
perda_pp_com_escudo = DIVIDA_TOTAL * 0.01 * (1 - ALIQUOTA_IR_CSLL)
perda_pp_sem_escudo = DIVIDA_TOTAL * 0.01

print(f"Perda por p.p. DENTRO do regime COM escudo fiscal (Selic até {selic_cobertura_ebit:.1%}): "
      f"R$ {perda_pp_com_escudo:,.0f} — constante em todo esse trecho.")
print(f"Perda por p.p. DENTRO do regime SEM escudo fiscal (Selic acima de {selic_cobertura_ebit:.1%}): "
      f"R$ {perda_pp_sem_escudo:,.0f} — constante em todo esse trecho.")
print("O 'desconto do IR' não enfraquece aos poucos — ele deixa de existir de uma vez, "
      "no momento em que o lucro tributável acaba.")

## 7. Animação: o lucro sumindo conforme a Selic sobe

O gráfico abaixo é interativo: use o botão ▶ (play) ou arraste o controle deslizante para ver, passo a passo (a cada 0,5 p.p. de Selic), a barra de **EBIT** ficar parada, a barra de **Despesa Financeira** crescer, e a barra de **Lucro Líquido** encolher até virar prejuízo.

In [ ]:
selic_frames = np.arange(5, 25.5, 0.5)
linhas_anim = []
for s in selic_frames:
    r = calcular_dre(s / 100)
    label_selic = f"{s:.1f}%"
    linhas_anim.append({"Selic": label_selic, "Selic_ord": s, "Categoria": "EBIT", "Valor (R$ mi)": r["EBIT (R$)"] / 1e6})
    linhas_anim.append({"Selic": label_selic, "Selic_ord": s, "Categoria": "Despesa Financeira", "Valor (R$ mi)": r["Despesa Financeira (R$)"] / 1e6})
    linhas_anim.append({"Selic": label_selic, "Selic_ord": s, "Categoria": "Lucro Líquido", "Valor (R$ mi)": r["Lucro Líquido (R$)"] / 1e6})

df_anim = pd.DataFrame(linhas_anim).sort_values("Selic_ord")

fig_anim = px.bar(
    df_anim, x="Categoria", y="Valor (R$ mi)", color="Categoria",
    animation_frame="Selic",
    range_y=[df_anim["Valor (R$ mi)"].min() - 2, df_anim["Valor (R$ mi)"].max() + 2],
    color_discrete_map={"EBIT": "#4c78a8", "Despesa Financeira": "#e45756", "Lucro Líquido": "#54a24b"},
    title="Quanto lucro desaparece conforme a Selic sobe (empresa fictícia, EBIT constante)",
)
fig_anim.update_layout(showlegend=False, height=520)
fig_anim.add_hline(y=0, line_dash="dash", line_color="black")
fig_anim.show()

## 8. Conclusão

- A despesa financeira desta dívida (CDI + spread) cresce de forma **linear** com a Selic. Como ela é subtraída de um EBIT que não muda, o lucro líquido cai de forma muito mais que proporcional — a Selic não precisa tocar a operação para destruir o resultado.
- O "juro é dedutível do IR" só reduz o custo **enquanto existe lucro tributável**, na alíquota efetiva e nas condições assumidas na seção 3. A perda de lucro por ponto percentual de Selic é **constante dentro de cada regime tributário** — mas salta para um patamar bem mais alto no exato momento em que o lucro tributável acaba (valores calculados na seção 6).
- O ponto em que a despesa financeira passa a cobrir 100% do EBIT (seção 5) é específico **desta** combinação de alavancagem e spread — não é um limite universal de Selic para qualquer empresa endividada. Mude os parâmetros da seção 3 para ver esse ponto se deslocar com os números da sua própria empresa.
- Uma empresa pode estar **operacionalmente saudável** (mesmas vendas, mesma margem, mesmos clientes) e ainda assim quebrar financeiramente, só porque a dívida foi contratada num regime de juros flutuantes sem qualquer proteção.
- Ferramentas de gestão de risco que ajudam a evitar esse cenário: reduzir a alavancagem, travar parte da dívida em taxa prefixada, contratar hedge (swap CDI x prefixado) e, principalmente, **rodar esse mesmo stress test com os números reais da sua empresa** antes de contratar a dívida, não depois.

---
*Simulação educacional com empresa e parâmetros fictícios (custo da dívida, alíquota de IR/CSLL e aproveitamento do tax shield declarados explicitamente na seção 3). Os dados de Selic histórica vêm da API pública do Banco Central (SGS, série 432). Este material não constitui recomendação de investimento ou de gestão financeira.*